# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Description:** This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:", getattr(metadata, 'name', 'N/A'))
print("Description:", getattr(metadata, 'description', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore available record sets by their @id
record_set_ids = [rs['@id'] for rs in metadata.recordSets] if hasattr(metadata, 'recordSets') else []

print('Record set @ids:')
for rid in record_set_ids:
    print(f"  - {rid}")

# Print the fields and column @ids for each record set
for rs in getattr(metadata, 'recordSets', []):
    print(f"\nRecord Set: {rs.get('@id', 'N/A')}")
    print("  Name:", rs.get('name', 'N/A'))
    print("  Description:", rs.get('description', 'N/A'))
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            field_id = field.get('@id', '')
            print(f"    - {field_id} | name: {field.get('name', '')} | column: {field.get('column', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Usually, scientific datasets provide one main record set. Let's select the first available record set.
# Replace the ID below with your dataset's record set @id as needed.
record_sets = record_set_ids
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    # If records are not empty, create a DataFrame. Else, continue.
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded {len(df)} records for record set: {record_set_id}")
        print("Columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records loaded for record set: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data, or grouping data for further analysis.

> **Note**: Update the `record_set_id` and field `@id`s below using the output of the previous cell.

In [ ]:
# --- EDA Example on Protein Abundance ---
# Please choose the appropriate record set and numeric field.

import numpy as np

# Assume we have a typical protein abundance table. Replace below with the correct IDs if needed.
# Example placeholders (set to real IDs based on output):
example_record_set_id = record_sets[0] if record_sets else None
example_df = dataframes[example_record_set_id] if example_record_set_id else None

if example_df is not None:
    # Attempt to auto-detect a numeric field for demonstration
    numeric_columns = example_df.select_dtypes(include=[np.number]).columns.tolist()
    print('Numeric fields:', numeric_columns)
    numeric_field = numeric_columns[0] if numeric_columns else None

    if numeric_field:
        threshold = example_df[numeric_field].mean()  # Use mean as arbitrary threshold
        filtered_df = example_df[example_df[numeric_field] > threshold].copy()
        print(f"\nFiltered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a likely protein identification field if exists (e.g., 'accession' or similar)
        # Try to find a categorical field with relatively high cardinality
        cat_fields = example_df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for field in cat_fields:
            n_unique = example_df[field].nunique()
            if n_unique < len(example_df) and n_unique > 1:
                group_field = field
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped means by field {group_field}:")
            display(grouped_df.head())
        else:
            print('No appropriate group field found.')
    else:
        print('No numeric fields available for EDA.')
else:
    print('No DataFrame loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> **Note**: Update the field names for the plot as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_df is not None and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(example_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If a categorical field exists, plot mean value by category (first 10 categories)
    if group_field:
        plt.figure(figsize=(10, 5))
        group_means = example_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrates how to:
- Load a scientific dataset defined by a Croissant schema with `mlcroissant`
- Discover record sets and their fields by their `@id`
- Extract tabular data and perform exploratory analysis
- Visualize distributions and group statistics

You can extend this notebook for further downstream analysis, such as machine learning modeling or specialized domain statistics, using the referenced field `@id`s for maximum schema traceability.